In [1]:
"""
This script is used to regress out principal components from gene expression of open chromatin data
authors: Roy Oelen
"""

'\nThis script is used to regress out principal components from gene expression of open chromatin data\nauthors: Roy Oelen\n'

In [3]:
# for checking file locations
import os
import glob
import os.path
# use warnings
import warnings
# use regex
import re
# get pandas plink info
from pandas_plink import read_plink1_bin
# for pandas
import pandas as pd
# do regression
from sklearn.linear_model import LinearRegression
# for multicore processing
import concurrent.futures

In [5]:
# location of the expression data
rna_data_loc = '/groups/umcg-franke-scrna/tmp04/projects/sc-eqtlgen-consortium-pipeline/ongoing/wg3/wg3_oneK1k/input/L1/Mono.qtlInput.txt'
# read the RNA data
rna_data = pd.read_csv(rna_data_loc, header = 0, sep = '\t', index_col = 0)

In [6]:
# location of the PCs
rna_pcs_loc = '/groups/umcg-franke-scrna/tmp04/projects/sc-eqtlgen-consortium-pipeline/ongoing/wg3/wg3_oneK1k/input/L1/Mono.qtlInput.Pcs.txt'
# read the PC data
rna_pcs = pd.read_csv(rna_pcs_loc, header = 0, sep = '\t', index_col = 0)

In [7]:
# get the samples present in both
rna_samples_both = list(set(rna_data.columns).intersection(rna_pcs.index))

In [8]:
# subset to the samples present in both
rna_data_matched = rna_data.loc[:, rna_samples_both]
rna_pcs_matched = rna_pcs.loc[rna_samples_both, ]

In [9]:
# create the model
model = LinearRegression()

# Create a new dataframe to store the residuals
rna_data_matched_residuals = pd.DataFrame(index=rna_data_matched.index, columns=rna_data_matched.columns)

# method to do a single gene
def regress_out_pc(gene):
    y = rna_data_matched.loc[gene].values
    X = rna_pcs_matched.values
    model.fit(X, y)
    residuals = y - model.predict(X)
    return gene, residuals

# do this concurrently
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(regress_out_pc, gene): gene for gene in rna_data_matched.index}
    # submit each job
    for future in concurrent.futures.as_completed(futures):
        gene, residuals = future.result()
        rna_data_matched_residuals.loc[gene] = residuals

In [10]:
# write the result
rna_residuas_loc = '/groups/umcg-franke-scrna/tmp04/projects/sc-eqtlgen-consortium-pipeline/ongoing/wg3/wg3_oneK1k/input/L1/Mono.qtlInput.PcCorrectedResiduals.txt.gz'
rna_data_matched_residuals.to_csv(rna_residuas_loc, sep = '\t', header = True, index = True, compression = 'gzip')

In [ ]:
# now for some atac data
atac_data_loc = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/qtl/caqtl/sc-eqtlgen/input/L1/UT/monocyte.qtlInput.txt.gz'
# read the atac data
atac_data = pd.read_csv(atac_data_loc, header = 0, sep = '\t', index_col = 0)
# location of the PCs
atac_pcs_loc = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/qtl/caqtl/sc-eqtlgen/input/L1/UT/monocyte.qtlInput.Pcs.txt.gz'
# read the PC data
atac_pcs = pd.read_csv(atac_pcs_loc, header = 0, sep = '\t', index_col = 0)
# get the samples present in both
atac_samples_both = list(set(atac_data.columns).intersection(atac_pcs.index))
# subset to the samples present in both
atac_data_matched = atac_data.loc[:, atac_samples_both]
atac_pcs_matched = atac_pcs.loc[atac_samples_both, ]
# create the model
model = LinearRegression()
# Create a new dataframe to store the residuals
atac_data_matched_residuals = pd.DataFrame(index=atac_data_matched.index, columns=atac_data_matched.columns)
# method to do a single gene
def regress_out_pc(gene):
    y = atac_data_matched.loc[gene].values
    X = atac_pcs_matched.values
    model.fit(X, y)
    residuals = y - model.predict(X)
    return gene, residuals

# do this concurrently
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(regress_out_pc, gene): gene for gene in atac_data_matched.index}
    # submit each job
    for future in concurrent.futures.as_completed(futures):
        gene, residuals = future.result()
        atac_data_matched_residuals.loc[gene] = residuals

# write the result
atac_residuals_loc = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/qtl/caqtl/sc-eqtlgen/input/L1/UT/monocyte.qtlInput.PcCorrectedResiduals.txt.gz'
atac_data_matched_residuals.to_csv(atac_residuals_loc, sep = '\t', header = True, index = True, compression = 'gzip')